# LoRA Fine-Tuning on Llama 3.2 1B — a hands-on walkthrough (Apple Silicon / MLX)

**Goal:** fine-tune a small model so it reliably outputs clean JSON for an extraction task, then *see* the before/after difference and measure it. By the end you'll have a real adapter, a merged model, and a true line you can defend on your resume.

**What you'll actually do:**
1. Run the base model on a task (the "before")
2. Build a tiny training dataset (the notebook makes it for you)
3. Train a LoRA adapter and watch the loss drop
4. Run the fine-tuned model on the same task (the "after")
5. Merge (fuse) the adapter into the base model
6. (Optional) convert to GGUF + quantize for serving — the same pipeline you already built

---
### ⚠️ Read before you run (your Mac runs warm)
Training uses the GPU harder than inference. This run is small (a 1B model, a few hundred steps, ~a few minutes), but:
- Close heavy apps first (browsers with many tabs, Docker, etc.).
- It's completely safe to **stop a cell early** (the ◼ button) — your adapter checkpoints are saved as it goes.
- If the fan gets loud, stop, let it cool, lower `--iters` or `--batch-size`, and resume. Nothing here *needs* to run long.

---
### Mental model recap
- **Prompting / RAG** = change what you *give* the model. Nothing inside it changes.
- **Fine-tuning** = change the model's *weights* so a skill is baked in.
- **LoRA** = freeze the big base model, train a tiny add-on "adapter" (<1% of the weights). Cheap, fast, and the adapter is just a few MB.
- **QLoRA** = LoRA but with the base model compressed to 4-bit to save memory. You only need it for *bigger* models; for 1B on 16 GB, plain **LoRA** is fine.

## 1. Install MLX-LM

`mlx-lm` is Apple's library for running and training LLMs on Apple Silicon (M1/M2/M3). It wraps everything we need: loading, generating, LoRA training, and fusing.

> If a command flag later doesn't match your version, run the `--help` cells — MLX moves fast and flag names occasionally change. The *concepts* stay identical.

In [1]:
%pip install -q "mlx-lm>=0.20.0"
print("installed")


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
installed


In [2]:
# Sanity check: MLX sees your Apple GPU
import mlx.core as mx
print("MLX default device:", mx.default_device())   # should say Device(gpu, 0) on M1/M2/M3

MLX default device: Device(gpu, 0)


## 2. Pick the base model

We use an MLX-ready copy of **Llama 3.2 1B Instruct**. The `mlx-community` org hosts pre-converted models so you don't convert anything yourself.

- `MODEL` below is the full-precision (bf16) version — best for plain LoRA.
- **If the download asks for authentication** (Llama is sometimes gated), either run `huggingface-cli login` in a terminal after accepting Meta's license on the model page, **or** just switch to the ungated Qwen fallback — the entire notebook works the same with it.

In [5]:
# Primary choice (Llama 3.2 1B, bf16):
MODEL = "mlx-community/Llama-3.2-1B-Instruct-bf16"

# Ungated fallback if Llama needs a login (Apache-2.0, works identically):
# MODEL = "mlx-community/Qwen2.5-1.5B-Instruct-bf16"

print("Using model:", MODEL)

Using model: mlx-community/Llama-3.2-1B-Instruct-bf16


## ⚙️ Environment setup — run this once (fixes the MPICH crash)

Your Jupyter kernel's Python is **Anaconda's**, and its `base` ships **MPICH**. MLX only supports **Open MPI**, so when MLX runs training it loads MPICH and **aborts** (the `exit 134` / kernel crash you saw). The clean fix, with **no changes to conda**: run every heavy MLX step (generate, train, score, fuse) in a tiny separate virtualenv built from your non-conda **python.org** Python, where MLX finds no MPI and just uses its single-process backend — each step in its own subprocess, so only one model is in memory at a time (no OOM on 16 GB).

Run the next cell once. It builds `./.venv-mlx` (~1 min the first time), installs `mlx-lm`, and defines the two helpers used everywhere below: `run_clean(...)` (runs an `mlx_lm` CLI) and `clean_py(...)` (runs our `mlx_tasks.py` script). It also needs `mlx_tasks.py` to sit next to this notebook.

In [ ]:
import os, subprocess

# Clean, non-conda Python to build the training venv from (python.org 3.13 on this machine).
BASE_PYTHON = "/Library/Frameworks/Python.framework/Versions/3.13/bin/python3"
CLEAN_PY = os.path.abspath(".venv-mlx/bin/python")

if not os.path.exists(CLEAN_PY):
    print("building clean training venv (.venv-mlx) - one time, ~1 min ...")
    subprocess.run([BASE_PYTHON, "-m", "venv", ".venv-mlx"], check=True)
    subprocess.run([CLEAN_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([CLEAN_PY, "-m", "pip", "install", "-q", "mlx-lm>=0.20.0"], check=True)

def _stream(cmd):
    """Run cmd in the clean venv, stream output live, return the captured text."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in proc.stdout:
        print(line, end=""); buf.append(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"subprocess failed (exit {proc.returncode})")
    return "".join(buf)

def run_clean(args):
    """Run an mlx_lm CLI module (e.g. 'mlx_lm.lora') in the clean venv."""
    return _stream([CLEAN_PY, "-m", *args])

def clean_py(args):
    """Run a script (e.g. 'mlx_tasks.py generate ...') in the clean venv. One model per call,
    loaded in an isolated subprocess and freed on exit -> safe on 16 GB."""
    return _stream([CLEAN_PY, *args])

print("training venv ready:", CLEAN_PY)

## 3. The "before" — how the base model behaves *untrained*

We give it an extraction task and ask for JSON. A raw instruct model usually *tries*, but often adds chatter ("Sure! Here is the JSON:..."), wrong keys, or markdown fences. That inconsistency is exactly what fine-tuning fixes.

`apply_chat_template` wraps your message in the special tokens the model was trained to expect (the `<|user|>` / `<|assistant|>` structure). Always use it for instruct models.

In [6]:
# "Before": run the BASE model in a short-lived clean-venv subprocess (one model, freed on exit).
TEST_TEXT = "Last quarter, Nimbus AI promoted Rahul Nair to staff ML engineer."
print("----- BASE MODEL (before fine-tuning) -----")
base_out = clean_py(["mlx_tasks.py", "generate", "--model", MODEL, "--text", TEST_TEXT])

----- BASE MODEL (before fine-tuning) -----


NameError: name 'clean_py' is not defined

## 4. Build a tiny training dataset

MLX-LM reads a folder with `train.jsonl` and `valid.jsonl`. We use the **chat format**: each line is one example with a `messages` list (your input + the ideal answer). MLX auto-detects this format.

Fine-tuning = showing the model many `input → ideal output` pairs. Here every ideal output is **strict JSON, no chatter** — so the model learns that *shape*. We generate ~200 varied examples programmatically (real projects use real labelled data; the format is identical).

In [ ]:
import json, random
from pathlib import Path
random.seed(0)

names = ["Priya Sharma","John Carter","Aisha Khan","Diego Alvarez","Mei Lin","Rahul Nair",
         "Sara Okoro","Tom Becker","Lena Petrova","Arjun Rao","Fatima Noor","Chris Evans"]
companies = ["Acme Corp","Globex","Initech","Umbrella Labs","Stark Industries","Hooli",
             "Pied Piper","Nimbus AI","Vertex Systems","Orbital Dynamics"]
roles = ["data scientist","backend engineer","product manager","ML engineer",
         "DevOps engineer","research scientist","software engineer","solutions architect"]
templates = [
    "{n} joined {c} last year as a {r}.",
    "At {c}, {n} works as a {r}.",
    "{n} is a {r} at {c}.",
    "We hired {n} as a {r} for {c} in 2024.",
    "{c} recently welcomed {n}, their new {r}.",
    "Last quarter, {c} promoted {n} to {r}.",
]

def make_example():
    n, c, r = random.choice(names), random.choice(companies), random.choice(roles)
    text = random.choice(templates).format(n=n, c=c, r=r)
    user = ("Extract name, company, and role from the text and return ONLY JSON "
            f"with keys name, company, role.\n\nText: {text}")
    assistant = json.dumps({"name": n, "company": c, "role": r})
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": assistant}]}

data = [make_example() for _ in range(200)]
train, valid = data[:170], data[170:]

Path("data").mkdir(exist_ok=True)
with open("data/train.jsonl", "w") as f:
    for ex in train: f.write(json.dumps(ex) + "\n")
with open("data/valid.jsonl", "w") as f:
    for ex in valid: f.write(json.dumps(ex) + "\n")

print(f"wrote {len(train)} train / {len(valid)} valid examples to ./data")
print("\none example:\n", json.dumps(train[0], indent=2))

## 5. Train the LoRA adapter

We call MLX-LM's LoRA trainer. First, peek at the flags for *your* installed version:

In [ ]:
import sys
!{sys.executable} -m mlx_lm.lora --help

### What the key hyperparameters mean
- **`--iters`** — how many training steps. More = learns more, runs longer/hotter. 200 is plenty here.
- **`--batch-size`** — examples per step. Bigger = faster but more memory. Keep small (2) on 16 GB.
- **`--num-layers`** — how many of the model's layers get a LoRA adapter. More layers = more capacity to learn, more memory. 8 is a good middle for a 1B.
- **`--learning-rate`** — how big each weight nudge is. `1e-4` is a safe default for LoRA.
- **`--adapter-path`** — where the trained adapter is saved.
- **`--steps-per-report` / `--steps-per-eval`** — how often it prints training loss / checks validation loss.

**Watch the `Train loss` number fall** as it runs — that's the model getting better at your task in real time. If validation loss starts *rising* while train loss keeps falling, that's overfitting (fine to ignore for this toy run).

In [ ]:
run_clean([
    "mlx_lm.lora",
    "--model", MODEL,
    "--train",
    "--data", "data",
    "--iters", "200",
    "--batch-size", "2",
    "--num-layers", "8",
    "--learning-rate", "1e-4",
    "--adapter-path", "adapters",
    "--steps-per-report", "20",
    "--steps-per-eval", "100",
])

## 6. The "after" — run the fine-tuned model

Same prompt as before, but now we load the base model **with the adapter on top** (`adapter_path=`). Compare this output to the base one from step 3 — it should now be clean JSON, no chatter.

In [ ]:
# "After": base model + your trained adapter, same prompt. Compare to the base output above.
print("----- FINE-TUNED (after) -----")
ft_out = clean_py(["mlx_tasks.py", "generate", "--model", MODEL, "--adapter", "adapters", "--text", TEST_TEXT])

print("\n=================== COMPARISON ===================")
print("BEFORE:\n", base_out.strip())
print("\nAFTER:\n", ft_out.strip())

## 7. Measure it (your real resume number)

The base instruct model already emits *valid* JSON, so that is too easy a bar. The real test is
**exact match**: does the output JSON equal the ground truth, including copying the role verbatim
(the base model likes to title-case `staff ML engineer` -> `Staff ML Engineer`, which is wrong)?

We score the base model and your fine-tuned model over the validation set -- each in its **own
subprocess**, so only one model is in memory at a time (no OOM). The gap between them is a true,
defensible number you can put on your resume.

In [ ]:
import re
print("scoring BASE model ...")
base_res = clean_py(["mlx_tasks.py", "score", "--model", MODEL, "--n", "20"])
print("\nscoring FINE-TUNED model ...")
ft_res = clean_py(["mlx_tasks.py", "score", "--model", MODEL, "--adapter", "adapters", "--n", "20"])

b = float(re.search(r"EXACT_MATCH_RATE ([\d.]+)", base_res).group(1))
f = float(re.search(r"EXACT_MATCH_RATE ([\d.]+)", ft_res).group(1))
print(f"\nBASE exact-match:       {b:.0%}")
print(f"FINE-TUNED exact-match: {f:.0%}")
print(f"\nResume-ready: fine-tuning raised exact structured-JSON match from {b:.0%} to {f:.0%}.")

## 8. Fuse the adapter into the model (optional)

So far you have: **base model + a small adapter file**. "Fusing" folds the adapter's changes back into the base weights, producing **one standalone model folder** — same size as the original base (fusing does *not* grow the parameter count).

Use this when you want to ship/serve a single model instead of base+adapter.

In [ ]:
run_clean([
    "mlx_lm.fuse",
    "--model", MODEL,
    "--adapter-path", "adapters",
    "--save-path", "llama32-1b-json",
])
print("fused model saved to ./llama32-1b-json")

## 9. (Optional, advanced) Serve it like your RAG app

Your fine-tuned model is a normal HF-format model now. To serve it fast the way you serve Llama 3.1 8B, you'd:

1. Convert it to **GGUF** with llama.cpp's `convert_hf_to_gguf.py` (you already have llama.cpp in your `local-document-assistant/serving/` project).
2. **Quantize** it (e.g. Q4_K_M) with `llama-quantize` — the exact step from your quantization case study.
3. Point `serve.sh` at the new GGUF.

That's the full circle: **fine-tune (here) → merge → GGUF → quantize → serve (your existing pipeline).** Nothing new to learn at the end — it's the skills you already have.

---
## Recap — what you just learned
- Fine-tuning changes the model's **weights**; LoRA does it cheaply via a tiny adapter.
- Dataset = `input → ideal output` pairs in chat JSONL; the *shape* of the outputs is what the model learns.
- You trained, watched loss drop, compared before/after, measured a real accuracy lift, and fused the result.
- QLoRA is the same thing with a 4-bit base — reach for it on bigger models / the RTX server.

**Once you've run this and have your real before→after number, ping me and we'll add the honest, specific bullet to your resume.**